# Day 3 · Exercise 5: Multi-Turn Conversations

**What you'll build:** `send_in_conversation` — a function that manages conversation history and lets the model remember what was said in previous turns.

**Why it matters:** The model itself is stateless — it only sees what you send. This function is the core of any multi-turn chatbot: you maintain the history, and the model reads the whole transcript each time.

## Your Implementation

In [ ]:
import ollama

MODEL = "llama3.2"

def send_in_conversation(
    history: list[dict],
    new_message: str,
) -> tuple[str, list[dict]]:
    """Send a new message in an ongoing conversation.

    Append new_message as a user turn to history, call the model with the
    full conversation, then append the model's reply as an assistant turn.

    Args:
        history:     Existing conversation as a list of {"role", "content"} dicts.
                     Pass [] to start a fresh conversation.
        new_message: The user's next message.

    Returns:
        A tuple (reply_text, updated_history) where updated_history has both
        the new user message and the model's reply appended.

    Example:
        reply, h = send_in_conversation([], "My name is Ada.")
        reply2, h = send_in_conversation(h, "What is my name?")
        # reply2 contains "Ada"
    """
    # ── YOUR CODE HERE ─────────────────────────────────────────
    pass
    # ───────────────────────────────────────────────────────────

## Check Your Work

Run the cell below — it runs 4 automated checks and shows ✅ / ❌ for each.

In [ ]:
_PASS, _FAIL = '✅', '❌'

def _ollama_running():
    try:
        import urllib.request  # stdlib — no install needed
        urllib.request.urlopen('http://localhost:11434/api/tags', timeout=3)
        return True
    except Exception:
        return False

def _run_checks():
    score, total = 0, 4

    # Check 1: function exists and is callable
    try:
        assert callable(send_in_conversation), 'send_in_conversation is not defined'
        print(f'{_PASS} Check 1/{total}: function exists and is callable')
        score += 1
    except Exception as e:
        print(f'{_FAIL} Check 1/{total}: {e}')
        return

    # Check 2: Ollama running + returns a (str, list) tuple
    if not _ollama_running():
        print(f'{_FAIL} Check 2/{total}: Ollama server is not running')
        print('  → macOS: open the Ollama app · Linux/Windows: ollama serve')
        return

    ret = None
    try:
        ret = send_in_conversation([], 'Hello! Just say hi back.')
        assert isinstance(ret, tuple) and len(ret) == 2, \
            f'expected a 2-tuple, got {type(ret).__name__}'
        reply, history = ret
        assert isinstance(reply, str), f'first element should be str, got {type(reply).__name__}'
        assert isinstance(history, list), f'second element should be list, got {type(history).__name__}'
        print(f'{_PASS} Check 2/{total}: returns (str, list) tuple')
        score += 1
    except AssertionError as e:
        print(f'{_FAIL} Check 2/{total}: {e}')
    except Exception as e:
        print(f'{_FAIL} Check 2/{total}: call failed — {e}')
        print('  → Is Ollama running? Is llama3.2 pulled?')
        return

    if ret is None:
        return

    reply, history = ret

    # Check 3: reply is non-empty
    try:
        assert len(reply) > 0, 'reply text is empty'
        print(f'{_PASS} Check 3/{total}: reply text is non-empty')
        score += 1
    except AssertionError as e:
        print(f'{_FAIL} Check 3/{total}: {e}')

    # Check 4: history grew by exactly 2 entries with correct roles
    try:
        assert len(history) == 2, \
            f'history should have 2 entries after one turn, got {len(history)}'
        assert history[0]['role'] == 'user', \
            f'first entry should have role "user", got "{history[0].get("role")}"'
        assert history[1]['role'] == 'assistant', \
            f'second entry should have role "assistant", got "{history[1].get("role")}"'
        print(f'{_PASS} Check 4/{total}: history has 2 entries with roles "user" then "assistant"')
        score += 1
    except AssertionError as e:
        print(f'{_FAIL} Check 4/{total}: {e}')

    print()
    if score == total:
        print('=' * 52)
        print(f'  {_PASS}  Exercise 5 complete! {total}/{total} checks passed.')
        print('=' * 52)
    else:
        print(f'  {score}/{total} passed. Keep going!')

_run_checks()

## Bonus Challenge

Test the model's memory over 3 turns:
```python
_, h = send_in_conversation([], "My favourite colour is indigo.")
_, h = send_in_conversation(h, "I also love the number 42.")
reply, h = send_in_conversation(h, "What is my favourite colour and number?")
print(reply)  # Should mention indigo and 42
print(f"History length: {len(h)}")  # Should be 6: 3 user + 3 assistant
```
The model has no memory of its own — it knows the answers because they're in the `history` list you've been passing in.

## Solution

<details>
<summary>Click to reveal — try on your own first</summary>

```python
import ollama

MODEL = "llama3.2"

def send_in_conversation(
    history: list[dict],
    new_message: str,
) -> tuple[str, list[dict]]:
    updated = history + [{"role": "user", "content": new_message}]
    response = ollama.chat(model=MODEL, messages=updated)
    reply = response["message"]["content"]
    updated = updated + [{"role": "assistant", "content": reply}]
    return reply, updated
```

**Why this works:** We use `history + [...]` (list concatenation) rather than `history.append(...)` to avoid mutating the caller's list — a safer pattern. After calling the model, we append the assistant reply to get the full updated history. Returning both the reply text and the history lets the caller display the reply immediately while storing the full context for the next turn. The model itself holds no state between calls — all the "memory" lives in `history`.
</details>